# Gemma-2 CBT Anxiety Support: train, serve, evaluate

**Part A** (any GPU runtime, T4 works): prepare CACTUS data, QLoRA fine-tune Gemma-2-2B-it, measure base vs fine-tuned perplexity, save the adapter to Google Drive.

**Part B** (fresh runtime, L4 / A100 recommended): serve Gemma-2 + the adapter and Llama Guard 3 with vLLM, start the FastAPI service, run the full-stack safety and response-quality evaluations.

Before you start:
1. Accept the licences on Hugging Face for `google/gemma-2-2b-it` and `meta-llama/Llama-Guard-3-1B`.
2. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) with a read token.

> **About the saved outputs:** the outputs below are copied from a Colab Tesla T4 run on 2026-11-04 (training logs trimmed to their final lines). Cells without output were run without their output being saved here, or have not been run yet.


## Part A: fine-tune

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
import os, subprocess
REPO_URL = "https://github.com/comet-web/gemma2-anxiety-support.git"
REPO_DIR = "/content/repo"
# Safe to re-run: clones once, then just pulls the latest commit.
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
os.chdir(REPO_DIR)
assert os.path.isfile("pyproject.toml"), f"repo not found in {os.getcwd()}"
print("working dir:", os.getcwd())
!pip install -q -e ".[train,dev]"
# Colab ships torchao 0.10; PEFT >= 0.21 refuses torchao < 0.16 and this project does not use it
!pip uninstall -y -q torchao

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(os.environ["HF_TOKEN"])

In [ ]:
# ~13k anxiety-focused CBT dialogues -> Gemma-2 chat format, train/eval/test split
!python training/prepare_data.py

Training time depends on the GPU. `max_train_samples: 4000` in `configs/train_qlora.yaml` is a few hours on a T4 and
much faster on an L4 / A100. For a quick first run pass `--max-train-samples 1000`.

In [ ]:
# 1) ~1 minute pipeline check on a tiny random Gemma-2: catches config/library errors before the long run
!python training/train_qlora.py --smoke
# 2) the real QLoRA run (add --max-train-samples 1000 for a quicker first pass)
!python training/train_qlora.py --config configs/train_qlora.yaml


[... --smoke run, final lines ...]
{'train_runtime': '2.035', 'train_samples_per_second': '1.966', 'train_steps_per_second': '0.983', 'train_loss': '12.45', 'epoch': '0.5'}
fine-tuned eval: {'eval_loss': 12.45190143585205, 'eval_model_preparation_time': 0.0023, 'eval_runtime': 0.0358, 'eval_samples_per_second': 111.759, 'eval_steps_per_second': 55.88, 'eval_entropy': 12.452807903289795, 'eval_mean_token_accuracy': 0.0, 'eval_num_tokens': 2048.0, 'epoch': 0.5}
{
  "model_id": "trl-internal-testing/tiny-Gemma2ForCausalLM",
  "gpu": "Tesla T4",
  "train_conversations": 8,
  "eval_conversations": 4,
  "train_runtime_min": 0.1,
  "train_loss": 12.452690124511719,
  "base_eval_loss": 12.451916694641113,
  "base_eval_perplexity": 255740.02871345408,
  "finetuned_eval_loss": 12.45190143585205,
  "finetuned_eval_perplexity": 255736.12646007302
}

[... real run (--max-train-samples 1000), final lines ...]
{'eval_loss': '1.092', 'eval_model_preparation_time': '0.0303', 'eval_runtime': '108.7', 'e

In [ ]:
import json
m = json.load(open("evaluation/results/training_metrics.json"))
print({k: v for k, v in m.items() if k != "config"})

In [ ]:
# Qualitative check with transformers + PEFT (base model vs fine-tuned adapter on the same prompt)
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

sys.path.insert(0, "src")
from anxiety_support.prompts import COUNSELOR_SYSTEM_PROMPT

base_id = "google/gemma-2-2b-it"
tok = AutoTokenizer.from_pretrained(base_id)

# Native bf16 needs compute capability >= 8 (A100/L4); a T4 (7.5) uses fp16
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    dtype = torch.bfloat16 if major >= 8 else torch.float16
else:
    dtype = torch.float32

base = AutoModelForCausalLM.from_pretrained(base_id, dtype=dtype, device_map="auto")
model = PeftModel.from_pretrained(base, "outputs/gemma2-cbt-qlora/adapter")
model.eval()

# Gemma-2 has no system role: the counselor instructions go in the first user message
msgs = [{
    "role": "user",
    "content": f"{COUNSELOR_SYSTEM_PROMPT}\n\nI have a presentation tomorrow and I'm sure everyone will think I'm stupid.",
}]

# transformers v5: apply_chat_template returns a dict (BatchEncoding)
inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
prompt_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    print("FINE-TUNED:\n", tok.decode(out[0, prompt_len:], skip_special_tokens=True))

    with model.disable_adapter():  # same weights, adapter switched off = base model
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    print("\nBASE:\n", tok.decode(out[0, prompt_len:], skip_special_tokens=True))

del model, base
if torch.cuda.is_available():
    torch.cuda.empty_cache()


FINE-TUNED:
 That sounds really tough. Can you tell me more about what's been going through your mind lately?

BASE:
 It sounds like you're feeling anxious about your presentation.  Let's explore that feeling a bit.  What's going through your mind right now when you think about the presentation? 


In [ ]:
# Persist the adapter + metrics to Drive for Part B
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/gemma2-cbt
!cp -r outputs/gemma2-cbt-qlora/adapter /content/drive/MyDrive/gemma2-cbt/
!cp evaluation/results/training_metrics.json /content/drive/MyDrive/gemma2-cbt/
# Optional: push the adapter to the Hub (private)
# !python training/merge_adapter.py --adapter outputs/gemma2-cbt-qlora/adapter --push-adapter <hf-user>/gemma2-2b-cbt-lora

## Part B: serve with vLLM and evaluate the full stack

**Runtime > Disconnect and delete runtime**, then start a fresh one (vLLM brings its own torch build) and run from here.

> **Status:** Part B has not completed yet. `meta-llama/Llama-Guard-3-1B` is a gated model and needs approved access on Hugging Face, otherwise the Llama Guard server fails with `403 ... not in the authorized list`. The Gemma-2 server and `eval_responses.py` only need the Gemma licence.


In [ ]:
import os, subprocess
REPO_URL = "https://github.com/comet-web/gemma2-anxiety-support.git"
REPO_DIR = "/content/repo"
# Safe to re-run: clones once, then just pulls the latest commit.
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
os.chdir(REPO_DIR)
assert os.path.isfile("pyproject.toml"), f"repo not found in {os.getcwd()}"
print("working dir:", os.getcwd())
!pip install -q vllm==0.32.0
!pip install -q -e ".[dev]"
import os
from google.colab import userdata, drive
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
drive.mount("/content/drive")
!mkdir -p outputs/gemma2-cbt-qlora && cp -r /content/drive/MyDrive/gemma2-cbt/adapter outputs/gemma2-cbt-qlora/
!mkdir -p evaluation/results && cp /content/drive/MyDrive/gemma2-cbt/training_metrics.json evaluation/results/
!python training/prepare_data.py   # recreates the held-out test split (same seed)

In [ ]:
# Start Llama Guard 3 (:8001), then Gemma-2 base + CBT LoRA (:8000), one after the other on one GPU.
# --enforce-eager skips torch.compile / CUDA-graph capture (slow and memory-hungry on a T4).
# If a server crashes, the last lines of its log are printed straight away.
import subprocess, time, urllib.request

def start_vllm(name, args, port, timeout_s=900):
    log = open(f"{name}.log", "w")
    proc = subprocess.Popen(["vllm", "serve", *args, "--port", str(port), "--max-model-len", "4096", "--enforce-eager"],
                            stdout=log, stderr=subprocess.STDOUT)
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            print(open(f"{name}.log").read()[-4000:])
            raise RuntimeError(f"{name} exited with code {proc.returncode}; log tail above")
        try:
            urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=2)
            print(f"{name} ready on :{port} after {time.time() - t0:.0f}s")
            return proc
        except Exception:
            time.sleep(5)
    print(open(f"{name}.log").read()[-4000:])
    raise RuntimeError(f"{name} not ready after {timeout_s}s; log tail above")

guard_proc = start_vllm("guard", ["meta-llama/Llama-Guard-3-1B", "--gpu-memory-utilization", "0.25"], 8001)
llm_proc = start_vllm("llm", ["google/gemma-2-2b-it", "--enable-lora", "--lora-modules",
                              "gemma-cbt=outputs/gemma2-cbt-qlora/adapter", "--max-lora-rank", "16",
                              "--gpu-memory-utilization", "0.55"], 8000)


In [ ]:
!nohup uvicorn anxiety_support.api.main:app --port 8080 > api.log 2>&1 &
!sleep 20
!curl -s localhost:8080/v1/chat -H 'content-type: application/json' -d '{"message": "I keep panicking before exams and my mind goes blank"}' | python -m json.tool

In [ ]:
# Safety ablation on the held-out prompt set
!python evaluation/eval_safety.py --target guard --split test
!python evaluation/eval_safety.py --target api --split test

In [ ]:
# Base vs fine-tuned response quality on held-out CACTUS conversations.
# Add --judge-model/--judge-base-url/--judge-api-key to use a stronger LLM judge (pairwise, both orderings).
!python evaluation/eval_responses.py --base-model google/gemma-2-2b-it --ft-model gemma-cbt -n 100

In [ ]:
# Save results so you can commit them to the repo
!cp -r evaluation/results /content/drive/MyDrive/gemma2-cbt/
!ls /content/drive/MyDrive/gemma2-cbt/results